# 01 — Data Preparation

This notebook merges, cleans, and structures the raw data for both the Gender and Race prediction pipelines.

**Data Sources:**
- **Gender:** U.S. Social Security Administration (SSA) baby name records, 1880–2021 (`data/raw/gender/names_till_2021/`)
- **Race:** U.S. Census Bureau surname–race probability table (`data/raw/race/Race_last_name.csv`)

**Outputs:**
- `data/processed/all_ssn_first_names_till_2021.csv` — full SSA dataset
- `data/processed/Unique_Names_Till_2021.csv` — deduplicated gender-labeled names
- `data/processed/Race_last_name.csv` — cleaned race-labeled surnames

In [ ]:
import os
import glob
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')

# ── Paths ────────────────────────────────────────────────────────────────────
BASE        = os.path.abspath(os.path.join(os.getcwd(), '..'))
RAW_GENDER  = os.path.join(BASE, 'data', 'raw', 'gender', 'names_till_2021')
RAW_RACE    = os.path.join(BASE, 'data', 'raw', 'race')
PROCESSED   = os.path.join(BASE, 'data', 'processed')

print('Base directory:', BASE)

## 1. Gender Data — Merge SSA Yearly Files

In [ ]:
# Load and concatenate all yobYYYY.txt files
yearly_files = sorted(glob.glob(os.path.join(RAW_GENDER, 'yob*.txt')))
print(f'Found {len(yearly_files)} yearly files  ({yearly_files[0][-12:]} → {yearly_files[-1][-12:]})')

frames = []
for fpath in yearly_files:
    year = int(os.path.basename(fpath)[3:7])
    df   = pd.read_csv(fpath, header=None, names=['Name', 'Gender', 'Frequency'])
    df['Year'] = year
    frames.append(df)

all_names = pd.concat(frames, ignore_index=True)
print(f'Total rows: {len(all_names):,}')
all_names.head()

In [ ]:
# Basic stats
print('Gender distribution (rows):')
print(all_names['Gender'].value_counts())
print(f'\nYear range: {all_names["Year"].min()} – {all_names["Year"].max()}')
print(f'Unique first names: {all_names["Name"].nunique():,}')

In [ ]:
# Save full merged dataset
out_all = os.path.join(PROCESSED, 'all_ssn_first_names_till_2021.csv')
if not os.path.exists(out_all):
    all_names.to_csv(out_all, index=False)
    print(f'Saved: {out_all}')
else:
    print(f'Already exists (skipped): {out_all}')

## 2. Gender Data — Deduplicate

In [ ]:
# Aggregate: sum frequencies across all years, keep majority gender
# A name like "Alex" may appear as both M and F across years; we keep the dominant one.
agg = (
    all_names
    .groupby(['Name', 'Gender'], as_index=False)['Frequency']
    .sum()
)

# Keep the gender with the highest total frequency per name
unique_names = (
    agg
    .sort_values('Frequency', ascending=False)
    .drop_duplicates(subset='Name', keep='first')
    .reset_index(drop=True)
)

print(f'Unique names: {len(unique_names):,}')
print('\nGender distribution (unique names):')
print(unique_names['Gender'].value_counts())
unique_names.head(10)

In [ ]:
out_unique = os.path.join(PROCESSED, 'Unique_Names_Till_2021.csv')
if not os.path.exists(out_unique):
    unique_names.to_csv(out_unique, index=False)
    print(f'Saved: {out_unique}')
else:
    print(f'Already exists (skipped): {out_unique}')
    unique_names = pd.read_csv(out_unique)
    print(f'Loaded {len(unique_names):,} rows from existing file')

## 3. Race Data — Load & Clean

In [ ]:
race_raw_path = os.path.join(RAW_RACE, 'Race_last_name.csv')
race_df = pd.read_csv(race_raw_path)
print(f'Raw race data shape: {race_df.shape}')
race_df.head()

In [ ]:
print('Columns:', race_df.columns.tolist())
print('\nMissing values per column:')
print(race_df.isnull().sum())

In [ ]:
# Identify name column and race column
# The CSV has columns: Name (or surname) and Race (category)
name_col = race_df.columns[0]
race_col = race_df.columns[1] if len(race_df.columns) > 1 else None

print(f'Name column : "{name_col}"')
print(f'Race column : "{race_col}"')

if race_col:
    print('\nRace categories:')
    print(race_df[race_col].value_counts())

In [ ]:
# Clean: standardize column names, drop duplicates, remove nulls
race_clean = race_df.copy()
race_clean.columns = [c.strip().lower().replace(' ', '_') for c in race_clean.columns]
race_clean = race_clean.dropna()
race_clean = race_clean.drop_duplicates(subset=race_clean.columns[0])
race_clean = race_clean.reset_index(drop=True)

print(f'Cleaned race data: {len(race_clean):,} unique surnames')

out_race = os.path.join(PROCESSED, 'Race_last_name.csv')
if not os.path.exists(out_race):
    race_clean.to_csv(out_race, index=False)
    print(f'Saved: {out_race}')
else:
    print(f'Already exists (skipped): {out_race}')
    race_clean = pd.read_csv(out_race)
    print(f'Loaded {len(race_clean):,} rows from existing file')

## 4. Exploratory Visualizations

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# — Gender distribution
gender_counts = unique_names['Gender'].value_counts()
axes[0].bar(gender_counts.index, gender_counts.values, color=['#4C72B0', '#DD8452'])
axes[0].set_title('Gender Distribution (Unique Names)', fontsize=13)
axes[0].set_xlabel('Gender')
axes[0].set_ylabel('Count')
for i, v in enumerate(gender_counts.values):
    axes[0].text(i, v + 200, f'{v:,}', ha='center', fontsize=10)

# — Race distribution
race_name_col = race_clean.columns[0]
race_label_col = race_clean.columns[1]
race_counts = race_clean[race_label_col].value_counts()
axes[1].barh(race_counts.index, race_counts.values, color='#55A868')
axes[1].set_title('Race Distribution (Unique Surnames)', fontsize=13)
axes[1].set_xlabel('Count')
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

plt.tight_layout()
plt.savefig(os.path.join(BASE, 'docs', 'data_distribution.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Name length distributions
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

unique_names['name_len'] = unique_names['Name'].str.len()
for g, col in [('F', '#DD8452'), ('M', '#4C72B0')]:
    subset = unique_names[unique_names['Gender'] == g]
    axes[0].hist(subset['name_len'], bins=range(1, 20), alpha=0.6, label=g, color=col)
axes[0].set_title('First-Name Length by Gender')
axes[0].set_xlabel('Name Length')
axes[0].set_ylabel('Count')
axes[0].legend()

race_clean['name_len'] = race_clean[race_name_col].str.len()
axes[1].hist(race_clean['name_len'], bins=range(1, 25), color='#55A868', alpha=0.7)
axes[1].set_title('Surname Length Distribution')
axes[1].set_xlabel('Name Length')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()
print('Data preparation complete.')